# 02. Baseline de Red Neuronal Recurrente (GRU)

En este segundo módulo, implementamos una arquitectura **GRU (Gated Recurrent Unit)** unidireccional utilizando **PyTorch**. 

El objetivo es establecer un punto de referencia (baseline) para la clasificación de emociones. Utilizaremos secuencias empaquetadas (*packed padded sequences*) para manejar eficientemente el padding y embeddings preentrenados de GloVe.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from datasets import load_dataset
import gensim.downloader as api

# Configuración de dispositivo (GPU si está disponible) y semillas para reproducibilidad
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Trabajando en: {DEVICE}')

### 1. Preparación de Datos

Definimos la función `map_fine_to_ekman` para agrupar las 28 emociones originales de Reddit en las 7 categorías universales de Ekman. Esto reduce la complejidad del problema y ayuda a mitigar el desbalanceo extremo de clases.

In [ ]:
def map_fine_to_ekman(dataset):
    """
    Mapea las etiquetas originales a categorías de Ekman.
    Explicación: Filtra las emociones finas y asigna la categoría de Ekman predominante 
    mediante un contador de frecuencias para cada instancia del dataset.
    """
    ekman2fine_mapping = {
        "anger": ["anger", "annoyance", "disapproval"], "disgust": ["disgust"], 
        "fear": ["fear", "nervousness"], "joy": ["joy", "amusement", "approval", "excitement", "gratitude", "love", "optimism", "relief", "pride", "admiration", "desire", "caring"],
        "sadness": ["sadness", "disappointment", "embarrassment", "grief", "remorse"], "surprise": ["surprise", "realization", "confusion", "curiosity"],
        "neutral": ["neutral"]
    }
    ekman2label = {cat: i for i, cat in enumerate(sorted(ekman2fine_mapping.keys()))}
    fine_to_ekman = {fine: ekman_cat for ekman_cat, fine_list in ekman2fine_mapping.items() for fine in fine_list}
    fine_labels = dataset["train"].features["labels"].feature.names

    def convert(example):
        fine_emos = [fine_labels[i] for i in example["labels"]]
        ekman_emos = [fine_to_ekman.get(e, "other") for e in fine_emos]
        # Se selecciona la emoción más común detectada en la frase
        chosen = Counter(ekman_emos).most_common(1)[0][0] if ekman_emos else "neutral"
        return {"text": example["text"], "labels": ekman2label[chosen]}

    return dataset.map(convert, remove_columns=dataset["train"].column_names)

ekman_dataset = map_fine_to_ekman(load_dataset("go_emotions", "simplified"))
print("Datos cargados y mapeados correctamente.")

### 2. Pipeline de Procesamiento de Texto y DataLoader

Para alimentar la red neuronal, necesitamos:
1. **Tokenizar**: Dividir el texto en palabras individuales.
2. **Vectorizar**: Usar el modelo **GloVe** para convertir cada palabra en un vector de 50 dimensiones.
3. **Gestionar Padding**: Las frases tienen longitudes distintas. Usamos una función `collate` personalizada para unificar la longitud de cada batch añadiendo ceros (*padding*) y guardando las longitudes reales para que la GRU no procese el relleno.

In [ ]:
print("Cargando modelo preentrenado GloVe (50d)...")
glove = api.load("glove-wiki-gigaword-50")
TOKEN_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)

def tokenize(text):
    """Limpia el texto y lo divide en tokens en minúsculas."""
    return [t.lower() for t in TOKEN_RE.findall(text)]

def make_collate_fn_gru(tokenizer_fn, glove_model):
    """
    Genera una función collate para el DataLoader.
    Explicación: Esta función procesa el batch en tiempo real, convirtiendo palabras a vectores
    GloVe, aplicando padding dinámico y devolviendo las longitudes reales para pack_padded_sequence.
    """
    emb_dim = glove_model.vector_size
    def collate(batch):
        texts, labels = zip(*[(d["text"], d["labels"]) for d in batch])
        seqs, lengths = [], []
        for text in texts:
            tokens = tokenizer_fn(text)
            vecs = [glove_model.get_vector(t) for t in tokens if t in glove_model]
            if not vecs: vecs = [np.zeros(emb_dim, dtype=np.float32)]
            arr = np.stack(vecs).astype(np.float32)
            seqs.append(torch.from_numpy(arr))
            lengths.append(arr.shape[0])
        
        # Padding manual hasta la frase más larga del batch
        padded = torch.zeros(len(seqs), max(lengths), emb_dim)
        for i, seq in enumerate(seqs): padded[i, :seq.shape[0]] = seq
        return padded, torch.tensor(lengths, dtype=torch.long), torch.tensor(labels, dtype=torch.long)
    return collate

collate_fn = make_collate_fn_gru(tokenize, glove)
train_loader = DataLoader(ekman_dataset["train"], batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(ekman_dataset["validation"], batch_size=64, collate_fn=collate_fn)
test_loader = DataLoader(ekman_dataset["test"], batch_size=64, collate_fn=collate_fn)

### 3. Definición de la Arquitectura GRU

Implementamos una clase `SentimentGRU` que hereda de `nn.Module`. 
La clave aquí es el uso de **`pack_padded_sequence`**: esto permite que la GRU ignore los vectores de ceros del padding, mejorando drásticamente el rendimiento y la precisión del modelo al enfocarse solo en contenido real.

In [ ]:
class SentimentGRU(nn.Module):
    def __init__(self, input_dim, hidden_size, num_classes, dropout=0.3):
        """
        input_dim: Dimensión de GloVe (50).
        hidden_size: Número de neuronas en la capa oculta.
        num_classes: 7 emociones.
        """
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_size, batch_first=True)
        # Capa de salida lineal
        self.fc = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, lengths):
        """
        Paso hacia adelante.
        Explicación: Empaqueta la secuencia, procesa con GRU, extrae el último estado oculto
        (representación final de la frase) y clasifica.
        """
        # Empaquetado para ignorar padding
        packed_input = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hn = self.gru(packed_input)
        # hn[-1] contiene el estado final de la secuencia
        return self.fc(self.dropout(hn[-1]))

### 4. Ciclo de Entrenamiento con Early Stopping

La función `train_model` gestiona el bucle de aprendizaje. Incluye un sistema de **Early Stopping** que detiene el entrenamiento si el error de validación no mejora tras 5 épocas, guardando siempre los mejores pesos encontrados.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=20, patience=5):
    """
    Entrena el modelo y monitoriza métricas.
    Explicación: Realiza retropropagación en Train y evaluación sin gradientes en Val.
    Devuelve un historial de métricas para análisis visual.
    """
    optimizer = optim.Adam(model.parameters(), lr=2e-3)
    criterion = nn.CrossEntropyLoss()
    best_val_loss = float('inf')
    counter = 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        # --- FASE DE ENTRENAMIENTO ---
        model.train()
        t_loss, t_corr, t_total = 0, 0, 0
        for x, lens, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x, lens)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            
            t_loss += loss.item() * x.size(0)
            t_corr += (out.argmax(1) == y).sum().item()
            t_total += x.size(0)
        
        # --- FASE DE VALIDACIÓN ---
        model.eval()
        v_loss, v_corr, v_total = 0, 0, 0
        with torch.no_grad():
            for x, lens, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x, lens)
                v_loss += criterion(out, y).item() * x.size(0)
                v_corr += (out.argmax(1) == y).sum().item()
                v_total += x.size(0)
        
        epoch_v_loss = v_loss / v_total
        history['train_loss'].append(t_loss/t_total); history['val_loss'].append(epoch_v_loss)
        history['train_acc'].append(t_corr/t_total); history['val_acc'].append(v_corr/v_total)
        
        print(f'E{epoch+1} | Loss: {t_loss/t_total:.3f} (T) {epoch_v_loss:.3f} (V) | Acc: {t_corr/t_total:.3f} (T) {v_corr/v_total:.3f} (V)')

        # Lógica de Early Stopping
        if epoch_v_loss < best_val_loss:
            best_val_loss = epoch_v_loss; counter = 0
            torch.save(model.state_dict(), 'best_gru.pt')
        else:
            counter += 1
            if counter >= patience:
                print("Early Stopping activado.")
                break
    return history

In [ ]:
model = SentimentGRU(50, 256, 7).to(DEVICE)
history = train_model(model, train_loader, val_loader)

# Evaluación Final en Test
model.load_state_dict(torch.load('best_gru.pt'))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, lens, y in test_loader:
        out = model(x.to(DEVICE), lens)
        y_true.extend(y.numpy())
        y_pred.extend(out.argmax(1).cpu().numpy())

print("\nClassification Report (Test):")
labels = sorted(["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"])
print(classification_report(y_true, y_pred, target_names=labels))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Matriz de Confusión - GRU Baseline')
plt.ylabel('Realidad')
plt.xlabel('Predicción')
plt.show()

## Análisis del Baseline

El modelo GRU logra un Accuracy de ~59%. A pesar de ser una mejora significativa respecto al azar (~14%), se observa un fuerte sesgo hacia la clase mayoritaria `joy`. 

**Conclusiones técnicas:**
1. La arquitectura GRU es capaz de captar la secuencia local, pero sufre con frases irónicas o muy cortas.
2. El desbalanceo de clases penaliza el F1-Score de categorías como `disgust` o `fear`.
3. Se confirma el solapamiento entre `surprise` y `neutral`, lo que sugiere que la atención contextual (Transformers) podría resolver mejor estas fronteras de decisión.